In [ ]:
# test_solution_prompt_1.py
import pytest
from solution import Scheduler, SharedObject

def test_scheduler_initialization():
    """Tests that the Scheduler can be created."""
    s = Scheduler()
    assert s is not None

def test_add_task_returns_unique_ids():
    """Tests that add() returns unique, sequential integer IDs."""
    s = Scheduler()
    def task_a(): yield
    def task_b(): yield
    assert s.add(task_a) == 0
    assert s.add(task_b) == 1

def test_run_single_task_to_completion():
    """Tests that the scheduler runs a single, simple task."""
    s = Scheduler()
    shared = SharedObject()
    shared.value = 0

    def task():
        shared.value = 1
        yield

    s.add(task)
    s.run()
    assert shared.value == 1

def test_cooperative_yielding_in_order():
    """Tests that tasks yield control and run in the order they were added."""
    s = Scheduler()
    shared = SharedObject()
    shared.value = []

    def task_a():
        shared.value.append("a1")
        yield
        shared.value.append("a2")

    def task_b():
        shared.value.append("b1")
        yield
        shared.value.append("b2")

    s.add(task_a)
    s.add(task_b)
    s.run()
    assert shared.value == ["a1", "b1", "a2", "b2"]

def test_tasks_with_arguments():
    """Tests that tasks can receive arguments correctly."""
    s = Scheduler()
    shared = SharedObject()
    shared.value = 0

    def adder_task(val, s_obj):
        s_obj.value += val
        yield

    s.add(adder_task, 5, shared)
    s.add(adder_task, 10, shared)
    s.run()
    assert shared.value == 15

def test_determinism_with_multiple_runs():
    """Tests that running the scheduler multiple times with the same tasks yields the same result."""
    shared = SharedObject()
    
    def create_and_run_scheduler():
        s = Scheduler()
        shared.value = []
        def task_a():
            shared.value.append(1)
            yield
            shared.value.append(2)
        def task_b():
            shared.value.append(3)
            yield
            shared.value.append(4)
        s.add(task_a)
        s.add(task_b)
        s.run()
        return shared.value

    result1 = create_and_run_scheduler()
    result2 = create_and_run_scheduler()
    assert result1 == result2
    assert result1 == [1, 3, 2, 4]

def test_run_with_no_tasks():
    """Tests that run() completes without error when no tasks are added."""
    s = Scheduler()
    try:
        s.run()
    except Exception as e:
        pytest.fail(f"Scheduler.run() with no tasks raised an exception: {e}")

def test_task_completes_in_one_go_if_no_yield():
    """Tests a task that runs to completion without yielding."""
    s = Scheduler()
    shared = SharedObject()
    shared.value = ""

    def task_a():
        shared.value += "a"
        if False: # This makes the function a generator without actually yielding
            yield

    def task_b():
        yield
        shared.value += "b"

    s.add(task_a)
    s.add(task_b)
    s.run()
    assert shared.value == "ab"



def test_add_rejects_non_generator_task():
    s = Scheduler()
    def not_a_generator():
        return 123
    with pytest.raises(TypeError):
        s.add(not_a_generator)

def test_add_accepts_function_returning_generator():
    s = Scheduler()
    # wrapper is not a generator function, but it returns a generator when called
    def inner():
        yield
    def wrapper():
        return inner()
    task_id = s.add(wrapper)
    assert isinstance(task_id, int)
    # Should run without errors
    s.run()

In [ ]:
import pytest
from solution import Scheduler, SharedObject


def _is_int_list(x):
    return isinstance(x, (list, tuple)) and all(isinstance(i, int) for i in x)


def test_run_returns_trace_object():
    """Tests that run() returns a sequence (trace in Prompt 2, outcomes in Prompt 3+)."""
    s = Scheduler()

    def task():
        yield

    s.add(task)
    result = s.run()
    assert result is not None
    assert isinstance(result, (list, tuple))


def test_replay_produces_identical_final_state():
    """Works as replay in Prompt 2; as outcome inclusion in Prompt 3+."""
    shared = SharedObject()

    def create_and_run():
        s = Scheduler()
        shared.value = []

        def task_a():
            shared.value.append("a1")
            yield
            shared.value.append("a2")

        def task_b():
            shared.value.append("b1")
            yield
            shared.value.append("b2")

        s.add(task_a)
        s.add(task_b)
        res = s.run()
        return res, list(shared.value)

    res, final_value_run = create_and_run()

    if hasattr(Scheduler, "replay") and _is_int_list(res):
        # Prompt 2 path: we have a trace and the shared value was mutated by run()
        assert final_value_run == ["a1", "b1", "a2", "b2"]

        # Now replay on a fresh scheduler and check we get identical final state
        s_replay = Scheduler()
        shared.value = []

        def task_a():
            shared.value.append("a1")
            yield
            shared.value.append("a2")

        def task_b():
            shared.value.append("b1")
            yield
            shared.value.append("b2")

        s_replay.add(task_a)
        s_replay.add(task_b)
        s_replay.replay(res)
        final_value_replay = shared.value
        assert final_value_run == final_value_replay
    else:
        # Prompt 3+ path: res is a list of final states; ensure expected outcome is present
        outs = res
        normalized = [v.value if hasattr(v, "value") else v for v in outs]
        assert ["a1", "b1", "a2", "b2"] in normalized


def test_replay_follows_exact_execution_path():
    """Exact replay in Prompt 2; outcome inclusion in Prompt 3+."""
    s_run = Scheduler()
    shared_run = SharedObject()
    shared_run.value = []

    def task_a(s_obj):
        s_obj.value.append("a")
        yield
        s_obj.value.append("c")

    def task_b(s_obj):
        s_obj.value.append("b")
        yield
        s_obj.value.append("d")

    s_run.add(task_a, shared_run)
    s_run.add(task_b, shared_run)
    res = s_run.run()

    if hasattr(Scheduler, "replay") and _is_int_list(res):
        assert shared_run.value == ["a", "b", "c", "d"]
    else:
        outs = res
        normalized = [v.value if hasattr(v, "value") else v for v in outs]
        assert ["a", "b", "c", "d"] in normalized

    s_run_2 = Scheduler()
    shared_run_2 = SharedObject()
    shared_run_2.value = []

    def task_x(s_obj):
        s_obj.value.append(1)
        yield
        s_obj.value.append(2)
        yield
        s_obj.value.append(3)

    def task_y(s_obj):
        s_obj.value.append(10)
        yield
        s_obj.value.append(20)

    s_run_2.add(task_x, shared_run_2)
    s_run_2.add(task_y, shared_run_2)
    res2 = s_run_2.run()

    if hasattr(Scheduler, "replay") and _is_int_list(res2):
        assert shared_run_2.value == [1, 10, 2, 20, 3]
        s_replay_2 = Scheduler()
        shared_replay_2 = SharedObject()
        shared_replay_2.value = []
        s_replay_2.add(task_x, shared_replay_2)
        s_replay_2.add(task_y, shared_replay_2)
        s_replay_2.replay(res2)
        assert shared_replay_2.value == [1, 10, 2, 20, 3]
    else:
        outs2 = res2
        normalized2 = [v.value if hasattr(v, "value") else v for v in outs2]
        assert [1, 10, 2, 20, 3] in normalized2


def test_replay_with_no_tasks():
    """Replay if available (Prompt 2); otherwise just verify run() returns a sequence."""
    s = Scheduler()
    result = s.run()
    if hasattr(Scheduler, "replay") and _is_int_list(result):
        s_replay = Scheduler()
        try:
            s_replay.replay(result)
        except Exception as e:
            pytest.fail(f"Replay with no tasks raised an exception: {e}")
    else:
        assert isinstance(result, (list, tuple))


def test_replay_is_independent_of_scheduler_instance():
    """Trace replays across instances in Prompt 2; single outcome check in Prompt 3+."""
    s1 = Scheduler()
    shared = SharedObject()
    shared.value = 0

    def task(s_obj):
        s_obj.value += 1
        yield
        s_obj.value *= 2

    s1.add(task, shared)
    res = s1.run()

    if hasattr(Scheduler, "replay") and _is_int_list(res):
        assert shared.value == 2
        s2 = Scheduler()
        shared2 = SharedObject()
        shared2.value = 0
        s2.add(task, shared2)
        s2.replay(res)
        assert shared2.value == 2
    else:
        # One task => exactly one outcome whose value must be 2
        outs = res
        normalized = [v.value if hasattr(v, "value") else v for v in outs]
        assert normalized == [2]

In [ ]:
# test_solution_prompt_3.py
import pytest
from solution import Scheduler, SharedObject

def test_run_returns_list_of_final_states():
    """Tests that run() returns a list."""
    s = Scheduler()
    def task(): yield
    s.add(task)
    result = s.run()
    assert isinstance(result, list)

def test_simple_interleaving_produces_all_unique_states():
    """Tests a simple scenario with two tasks to ensure all outcomes are found."""
    s = Scheduler()
    s_obj = SharedObject()
    s_obj.value = ""
    def task_A(s):
        s.value += "A"
        yield

    def task_B(s):
        s.value += "B"
        yield

    s.add(task_A, s_obj)
    s.add(task_B, s_obj)
    
    results = s.run()
    final_values = sorted([r.value for r in results])
    assert final_values == ["AB", "BA"]

def test_complex_interleaving_all_outcomes():
    """Tests a non-atomic operation to find all possible outcomes."""
    s = Scheduler()
    s_obj = SharedObject()
    
    def non_atomic_inc(shared_obj):
        # This operation is intentionally racy: read, yield, write
        current_val = shared_obj.value
        yield
        shared_obj.value = current_val + 1

    s.add(non_atomic_inc, s_obj)
    s.add(non_atomic_inc, s_obj)

    s_obj.value = 0
    results = s.run()

    final_values = sorted([r.value for r in results])
    assert final_values == [1, 2]

def test_determinism_of_interleaving_results():
    """Ensures the list of final states is returned in a deterministic order."""
    def run_once():
        s = Scheduler()
        s_obj = SharedObject()
        s_obj.value = ""
        def task_A(s): s.value += "A"; yield
        def task_B(s): s.value += "B"; yield
        s.add(task_A, s_obj)
        s.add(task_B, s_obj)
        return [r.value for r in s.run()]

    values1 = run_once()
    values2 = run_once()
    assert values1 == values2

def test_interleaving_with_different_task_lengths():
    """Tests exploration when tasks have a different number of yield points."""
    s = Scheduler()
    s_obj = SharedObject()

    def task_A(s):
        s.value += 'A1'
        yield
        s.value += 'A2'

    def task_B(s):
        s.value += 'B1'
        # Make this a generator that runs to completion immediately
        if False: yield

    s.add(task_A, s_obj)
    s.add(task_B, s_obj)

    s_obj.value = ""
    results = s.run()
    final_values = sorted([r.value for r in results])
    
    # All legal interleavings are valid:
    expected = sorted(["A1A2B1", "A1B1A2", "B1A1A2"])
    assert final_values == expected

def test_no_tasks_returns_empty_list():
    """Tests that run() returns an empty list if no tasks are added."""
    s = Scheduler()
    assert s.run() == []

In [ ]:
# test_solution_prompt_4.py
import pytest
from solution import Scheduler, SharedObject, Lock

def _get_attr(state, name):
    """Flexible accessor: try attribute, dict key, or state.value (dict)"""
    if hasattr(state, name):
        return getattr(state, name)
    if isinstance(state, dict) and name in state:
        return state[name]
    v = getattr(state, "value", None)
    if isinstance(v, dict) and name in v:
        return v[name]
    raise AssertionError(f"final state does not expose attribute {name!r}: {state!r}")

def _to_hashable(x):
    """Recursively convert common container types to an immutable/hashable representation."""
    if isinstance(x, (int, float, str, bool, type(None))):
        return x
    if isinstance(x, dict):
        return tuple((k, _to_hashable(v)) for k, v in sorted(x.items(), key=lambda p: str(p[0])))
    if isinstance(x, (list, tuple)):
        return tuple(_to_hashable(v) for v in x)
    if isinstance(x, set):
        return tuple(sorted((_to_hashable(v) for v in x), key=repr))
    return repr(x)

def test_lock_mutual_exclusion():
    """Tests that a lock correctly enforces mutual exclusion for a simple increment."""
    s = Scheduler()
    shared = SharedObject()
    # Initialize a counter; accept attribute or nested dict shape
    try:
        shared.value = 0
    except Exception:
        shared.value = {'value': 0}

    lock = Lock()

    def critical_section_task(s_obj, lk):
        yield from lk.acquire()
        curr = _get_attr(s_obj, "value")
        # force a scheduling point while holding the lock
        yield
        # write back
        if hasattr(s_obj, "value") and not isinstance(getattr(s_obj, "value"), dict):
            s_obj.value = curr + 1
        else:
            v = getattr(s_obj, "value")
            if isinstance(v, dict):
                v['value'] = curr + 1
            else:
                s_obj.value = curr + 1
        yield from lk.release()

    s.add(critical_section_task, shared, lock)
    s.add(critical_section_task, shared, lock)

    results = s.run()

    final_values = set()
    for r in results:
        val = _get_attr(r, "value")
        final_values.add(_to_hashable(val))
    assert final_values == {_to_hashable(2)}

def test_lock_blocking_and_unblocking():
    """Tests that tasks block on acquire and resume on release; ensure mutual exclusion holds."""
    s = Scheduler()
    shared = SharedObject()
    # allow either attribute or nested dict for 'log'
    try:
        shared.log = []
    except Exception:
        shared.value = {'log': []}
    lock = Lock()

    def append_log(s_obj, msg):
        if hasattr(s_obj, "log"):
            s_obj.log.append(msg)
        else:
            s_obj.value['log'].append(msg)

    def task_a(s_obj, lk):
        append_log(s_obj, "A:acq")
        yield from lk.acquire()
        append_log(s_obj, "A:in")
        # LOG BEFORE ACTUAL RELEASE so logs bracket the critical section
        append_log(s_obj, "A:rel")
        yield from lk.release()

    def task_b(s_obj, lk):
        append_log(s_obj, "B:acq")
        yield from lk.acquire()
        append_log(s_obj, "B:in")
        # LOG BEFORE ACTUAL RELEASE so logs bracket the critical section
        append_log(s_obj, "B:rel")
        yield from lk.release()

    s.add(task_a, shared, lock)
    s.add(task_b, shared, lock)

    results = s.run()

    # Extract logs for each final snapshot; tolerant to several snapshot shapes
    final_logs = set()
    for r in results:
        try:
            logs = _get_attr(r, "log")
        except AssertionError:
            v = getattr(r, "value", None)
            if isinstance(v, dict) and 'log' in v:
                logs = v['log']
            else:
                raise
        final_logs.add(tuple(logs))

    def check_mutual_exclusion(log_seq):
        # find positions
        def idx_of(item):
            try:
                return log_seq.index(item)
            except ValueError:
                return None
        A_in = idx_of("A:in"); A_rel = idx_of("A:rel")
        B_in = idx_of("B:in"); B_rel = idx_of("B:rel")
        assert A_in is not None and A_rel is not None and A_in < A_rel
        assert B_in is not None and B_rel is not None and B_in < B_rel
        # with rel logged before releasing, the intervals must not overlap
        overlap = (A_in < B_in < A_rel) or (B_in < A_in < B_rel)
        assert not overlap, f"critical sections overlapped in log: {log_seq!r}"

    for seq in final_logs:
        check_mutual_exclusion(list(seq))

def test_releasing_unacquired_lock_is_handled_gracefully_or_errors():
    """
    The prompt did not mandate exact behavior for releasing a lock not held by the caller.
    Accept either:
      - Scheduler.run() raises an exception (implementation chose to treat it as an error), OR
      - the scheduler completes without crashing (implementation chose to ignore/handle it).
    """
    s = Scheduler()
    lock = Lock()

    def bad_release_task(lk):
        # attempt to release without acquiring
        yield from lk.release()

    s.add(bad_release_task, lock)
    try:
        res = s.run()
    except Exception:
        # acceptable: implementation treats release-by-non-owner as an error
        return
    else:
        # acceptable: implementation handled the call; ensure the run completed and returned results
        if isinstance(res, tuple):
            final_states = res[0]
        else:
            final_states = res
        assert isinstance(final_states, list)

def test_multiple_locks():
    """Tests interleavings with two different locks. Final counters a and b must be 2."""
    s = Scheduler()
    shared = SharedObject()
    # set attributes expected by the test
    try:
        shared.a = 0
        shared.b = 0
    except Exception:
        shared.value = {'a': 0, 'b': 0}

    lock_a = Lock()
    lock_b = Lock()

    def task_1(s_obj, lk_a, lk_b):
        yield from lk_a.acquire()
        if hasattr(s_obj, "a"):
            s_obj.a += 1
        else:
            s_obj.value['a'] += 1
        yield from lk_a.release()

        yield from lk_b.acquire()
        if hasattr(s_obj, "b"):
            s_obj.b += 1
        else:
            s_obj.value['b'] += 1
        yield from lk_b.release()

    def task_2(s_obj, lk_a, lk_b):
        yield from lk_b.acquire()
        if hasattr(s_obj, "b"):
            s_obj.b += 1
        else:
            s_obj.value['b'] += 1
        yield from lk_b.release()

        yield from lk_a.acquire()
        if hasattr(s_obj, "a"):
            s_obj.a += 1
        else:
            s_obj.value['a'] += 1
        yield from lk_a.release()

    s.add(task_1, shared, lock_a, lock_b)
    s.add(task_2, shared, lock_a, lock_b)

    results = s.run()

    tuples = set()
    for r in results:
        a_val = _get_attr(r, "a")
        b_val = _get_attr(r, "b")
        tuples.add((_to_hashable(a_val), _to_hashable(b_val)))
        assert _to_hashable(a_val) == _to_hashable(2)
        assert _to_hashable(b_val) == _to_hashable(2)

    assert tuples == {(_to_hashable(2), _to_hashable(2))}

In [ ]:
# test_solution_prompt_5.py
import pytest
from solution import Scheduler, SharedObject, Lock

def test_dpor_produces_same_states_as_exhaustive_search():
    """
    Tests that DPOR optimization returns the exact same set of unique final states
    as a naive exhaustive search would for a simple locking scenario.
    """
    s = Scheduler()
    shared = SharedObject()
    lock = Lock()
    shared.value = ""

    def task_a(s_obj, lk):
        yield from lk.acquire()
        s_obj.value += "A"
        yield from lk.release()

    def task_b(s_obj, lk):
        yield from lk.acquire()
        s_obj.value += "B"
        yield from lk.release()

    s.add(task_a, shared, lock)
    s.add(task_b, shared, lock)

    results = s.run()
    final_values = sorted([r.value for r in results])
    
    # The two serializations are "AB" and "BA". DPOR must not prune one of them.
    assert final_values == ["AB", "BA"]

def test_dpor_on_independent_operations():
    """
    Tests a scenario with independent operations where DPOR should offer significant pruning.
    The test verifies that only one final state is reported, as all paths are equivalent.
    """
    s = Scheduler()
    shared = SharedObject()
    shared.a = 0
    shared.b = 0

    def task_inc_a(s_obj):
        # This is a non-atomic increment, but on a separate variable.
        # It's intentionally racy if used on the same variable, but independent here.
        val = s_obj.a
        yield # Yield to allow interleaving
        s_obj.a = val + 1
        

    def task_inc_b(s_obj):
        val = s_obj.b
        yield # Yield to allow interleaving
        s_obj.b = val + 1
        

    s.add(task_inc_a, shared)
    s.add(task_inc_b, shared)

    results = s.run()
    
    # Without DPOR, this would explore interleavings like:
    # A reads a, B reads b, A writes a, B writes b -> (1, 1)
    # A reads a, B reads b, B writes b, A writes a -> (1, 1)
    # etc. All lead to the same final state.
    # DPOR should recognize the independence of operations on `a` and `b`
    # and explore only one representative schedule.
    # The key is that we must end up with the correct and *only* the correct final state.
    assert len(results) == 1
    final_state = results[0]
    assert final_state.a == 1
    assert final_state.b == 1

def test_dpor_complex_case_with_locks():
    """
    A more complex test where some operations are independent and some are dependent.
    T1: inc(a), lock(L), inc(c), unlock(L)
    T2: inc(b), lock(L), inc(c), unlock(L)
    """
    s = Scheduler()
    shared = SharedObject()
    shared.a = 0
    shared.b = 0
    shared.c = 0
    lock = Lock()
    
    def task_1(s_obj, lk):
        # Independent part
        s_obj.a += 1
        
        # Dependent part
        yield from lk.acquire()
        s_obj.c += 1
        yield from lk.release()
        
    def task_2(s_obj, lk):
        # Independent part
        s_obj.b += 1

        # Dependent part
        yield from lk.acquire()
        s_obj.c += 1
        yield from lk.release()

    s.add(task_1, shared, lock)
    s.add(task_2, shared, lock)

    results = s.run()

    # The updates to a and b are independent. The updates to c are serialized by the lock.
    # Every possible final state must have a=1, b=1, and c=2.
    # Because all paths lead to the same outcome, DPOR should find only one.
    assert len(results) == 1
    final_state = results[0]
    assert final_state.a == 1
    assert final_state.b == 1
    assert final_state.c == 2

In [ ]:
# test_solution_prompt_6.py
import pytest
from solution import Scheduler, SharedObject, Channel

def test_channel_send_receive_simple():
    """Tests a single value sent and received between two tasks."""
    s = Scheduler()
    shared = SharedObject()
    shared.received_value = None
    # Buffer of 1 allows send to complete before receive starts
    channel = Channel(capacity=1)

    def sender(ch):
        yield from ch.send(42)

    def receiver(ch, s_obj):
        val = yield from ch.receive()
        s_obj.received_value = val

    s.add(sender, channel)
    s.add(receiver, channel, shared)

    results = s.run()
    # There should be one unique outcome where the value is successfully transferred.
    assert len(results) == 1
    assert results[0].received_value == 42

def test_channel_blocking_on_full_buffer():
    """Tests that a sender blocks when the channel buffer is full."""
    s = Scheduler()
    shared = SharedObject()
    shared.log = []
    # No buffer, send will block until receive is ready
    channel = Channel(capacity=0)

    def sender(ch, s_obj):
        s_obj.log.append("sender:sending")
        yield from ch.send(1)
        s_obj.log.append("sender:sent")

    def receiver(ch, s_obj):
        s_obj.log.append("receiver:receiving")
        yield from ch.receive()
        s_obj.log.append("receiver:received")

    s.add(sender, channel, shared)
    s.add(receiver, channel, shared)
    
    results = s.run()

    # With a capacity of 0, a send must wait for a receive.
    # The only valid interleaving of events is:
    # Path 1: sender starts send, receiver starts receive, rendezvous, sender continues, receiver continues.
    #   - "sender:sending", "receiver:receiving", "sender:sent", "receiver:received"
    # Path 2: receiver starts receive, sender starts send, rendezvous, ...
    #   - "receiver:receiving", "sender:sending", "sender:sent", "receiver:received"
    # Let's check the set of possible outcomes.
    
    expected_logs = {
        ("sender:sending", "receiver:receiving", "sender:sent", "receiver:received"),
        ("receiver:receiving", "sender:sending", "sender:sent", "receiver:received"),
    }
    
    final_logs = {tuple(r.log) for r in results}
    assert final_logs == expected_logs

def test_channel_blocking_on_empty_buffer():
    """Tests that a receiver blocks when the channel is empty."""
    s = Scheduler()
    shared = SharedObject()
    shared.log = []
    channel = Channel(capacity=1)

    def receiver(ch, s_obj):
        s_obj.log.append("receiver:receiving")
        yield from ch.receive()
        s_obj.log.append("receiver:received")

    def sender(ch, s_obj):
        s_obj.log.append("sender:sending")
        yield from ch.send(1)
        s_obj.log.append("sender:sent")

    # Add receiver first to encourage it to run first
    s.add(receiver, channel, shared)
    s.add(sender, channel, shared)
    
    results = s.run()
    
    # Receiver must block until sender sends.
    # The only valid sequence is:
    # "receiver:receiving", "sender:sending", "sender:sent", "receiver:received"
    
    final_logs = {tuple(r.log) for r in results}
    assert final_logs == {("receiver:receiving", "sender:sending", "sender:sent", "receiver:received")}

def test_multiple_items_in_buffered_channel():
    """Tests sending and receiving multiple items through a buffered channel."""
    s = Scheduler()
    shared = SharedObject()
    shared.received = []
    channel = Channel(capacity=2)

    def sender(ch):
        yield from ch.send(10)
        yield from ch.send(20)

    def receiver(ch, s_obj):
        v1 = yield from ch.receive()
        v2 = yield from ch.receive()
        s_obj.received = [v1, v2]

    s.add(sender, channel)
    s.add(receiver, channel, shared)
    
    results = s.run()
    # All paths should result in the receiver getting [10, 20]
    final_values = {tuple(r.received) for r in results}
    assert final_values == {(10, 20)}

In [ ]:
# test_solution_prompt_7.py
import pytest
from solution import Scheduler, SharedObject, Lock, Channel

def test_run_return_signature_for_no_deadlock():
    """Tests that run() returns the correct tuple format when no deadlock occurs."""
    s = Scheduler()
    def task(): yield
    s.add(task)
    result = s.run()
    assert isinstance(result, tuple)
    assert len(result) == 2
    final_states, deadlock_traces = result
    assert isinstance(final_states, list)
    assert isinstance(deadlock_traces, list)
    assert len(deadlock_traces) == 0

def test_simple_deadlock_detection():
    """Tests detection of a simple AB-BA deadlock."""
    s = Scheduler()
    lock_a = Lock()
    lock_b = Lock()

    def task_1(lk_a, lk_b):
        yield from lk_a.acquire()
        yield from lk_b.acquire() # Will block waiting for task_2
        yield from lk_a.release()
        yield from lk_b.release()

    def task_2(lk_a, lk_b):
        yield from lk_b.acquire()
        yield from lk_a.acquire() # Will block waiting for task_1
        yield from lk_b.release()
        yield from lk_a.release()

    s.add(task_1, lock_a, lock_b)
    s.add(task_2, lock_a, lock_b)

    final_states, deadlock_traces = s.run()
    
    # This program can only deadlock. No final states should be found.
    assert len(final_states) == 0
    # Exactly one deadlock scenario should be found.
    assert len(deadlock_traces) == 1
    # The trace should be a non-empty object.
    assert deadlock_traces[0] is not None

def test_no_deadlock_with_ordered_locks():
    """Tests that acquiring locks in a consistent order prevents deadlocks."""
    s = Scheduler()
    lock_a = Lock()
    lock_b = Lock()
    shared = SharedObject()
    shared.value = 0

    def task(lka, lkb, s_obj):
        yield from lka.acquire()
        yield from lkb.acquire()
        s_obj.value += 1
        yield from lkb.release()
        yield from lka.release()

    s.add(task, lock_a, lock_b, shared)
    s.add(task, lock_a, lock_b, shared)
    
    final_states, deadlock_traces = s.run()
    
    assert len(deadlock_traces) == 0
    assert len(final_states) == 1
    assert final_states[0].value == 2

def test_deadlock_with_channels():
    """Tests a deadlock scenario involving channels."""
    s = Scheduler()
    ch1 = Channel(capacity=0)
    ch2 = Channel(capacity=0)

    def task_a(c1, c2):
        yield from c1.send(1)      # Blocks until B receives
        val = yield from c2.receive() # Blocks until B sends

    def task_b(c1, c2):
        val = yield from c1.receive() # Blocks until A sends
        yield from c2.send(val)   # Blocks until A receives
    
    # This setup is fine. Let's make it deadlock.
    # A sends on c1, waits for receive on c2.
    # B sends on c2, waits for receive on c1.
    def deadlock_task_a(c1, c2):
        yield from c1.send(1)
        yield from c2.receive()

    def deadlock_task_b(c1, c2):
        yield from c2.send(2)
        yield from c1.receive()

    s.add(deadlock_task_a, ch1, ch2)
    s.add(deadlock_task_b, ch1, ch2)

    final_states, deadlock_traces = s.run()
    assert len(final_states) == 0
    assert len(deadlock_traces) == 1

In [ ]:
# test_solution_prompt_8.py
import pytest
from solution import Scheduler, SharedObject

# We assume the user's SharedObject is now instrumented or that the scheduler
# can wrap it to detect reads/writes.
# For testing, we can define our own instrumented object if the prompt allows.
# The prompt says "Modify the SharedObject", so we assume the user's code has it.

def test_run_return_signature_for_no_races():
    """Tests the new 3-tuple return signature in a race-free program."""
    s = Scheduler()
    def task(): yield
    s.add(task)
    result = s.run()
    assert isinstance(result, tuple)
    assert len(result) == 3
    final_states, deadlock_traces, race_reports = result
    assert isinstance(final_states, list)
    assert isinstance(deadlock_traces, list)
    assert isinstance(race_reports, list)
    assert len(race_reports) == 0

def test_simple_data_race_detection():
    """Tests detection of a simple read-write race."""
    s = Scheduler()
    shared = SharedObject()
    shared.value = 0

    def task_writer(s_obj):
        s_obj.value = 1 # Write access
        yield

    def task_reader(s_obj):
        _ = s_obj.value # Read access
        yield

    s.add(task_writer, shared)
    s.add(task_reader, shared)
    
    final_states, deadlock_traces, race_reports = s.run()
    
    # This program has a race because the read and write can happen concurrently.
    assert len(race_reports) >= 1 # Could be one for each interleaving
    # Let's check for at least one valid report
    report = race_reports[0]
    # A report should identify the tasks and provide a trace.
    assert 'trace' in dir(report) or isinstance(report, tuple) # Flexible report format
    
def test_write_write_race_detection():
    """Tests detection of a write-write race."""
    s = Scheduler()
    shared = SharedObject()
    shared.value = 0

    def writer_1(s_obj):
        s_obj.value = 10
        yield

    def writer_2(s_obj):
        s_obj.value = 20
        yield

    s.add(writer_1, shared)
    s.add(writer_2, shared)

    final_states, deadlock_traces, race_reports = s.run()
    assert len(race_reports) >= 1

def test_no_race_with_synchronization():
    """Tests that no race is reported when access is protected by a lock."""
    # This requires the solution to support both Locks and race detection.
    # We need to import Lock for the test.
    from solution import Lock

    s = Scheduler()
    shared = SharedObject()
    shared.value = 0
    lock = Lock()
    
    def task(s_obj, lk):
        yield from lk.acquire()
        s_obj.value += 1
        yield from lk.release()
        
    s.add(task, shared, lock)
    s.add(task, shared, lock)
    
    final_states, deadlock_traces, race_reports = s.run()

    assert len(race_reports) == 0
    assert len(deadlock_traces) == 0
    assert len(final_states) > 0 # Should complete successfully

def test_no_race_on_different_objects():
    """Tests that no race is reported for accesses to different SharedObjects."""
    s = Scheduler()
    shared1 = SharedObject()
    shared1.value = 0
    shared2 = SharedObject()
    shared2.value = 0
    
    def task1(s1):
        s1.value = 1
        yield
        
    def task2(s2):
        s2.value = 2
        yield
        
    s.add(task1, shared1)
    s.add(task2, shared2)
    
    _f, _d, races = s.run()
    assert len(races) == 0

In [ ]:
# test_solution_prompt_9.py
import pytest
from solution import Scheduler, SharedObject, Channel

def test_state_hashing_terminates_on_cycle():
    """
    Tests that the scheduler can terminate when exploring a program with a state cycle.
    Without state hashing, this would run forever.
    """
    s = Scheduler()
    shared = SharedObject()
    shared.value = 0
    channel = Channel(capacity=1)

    def loopy_task(s_obj, ch):
        # This task creates a cycle: it increments a value, sends it,
        # then receives and does it again.
        # With two such tasks, they can feed each other forever.
        # But the global state (pc, s_obj.value, channel state) will repeat.
        
        # To make the state finite, let's use a small integer range.
        my_val = 1
        while True:
            yield from ch.send(my_val)
            received_val = yield from ch.receive()
            my_val = (received_val % 2) + 1 # Keep value in {1, 2}
    
    # Prime the channel to start the loop
    s.add(lambda ch: ch.send(1), channel) 
    
    s.add(loopy_task, shared, channel)
    s.add(loopy_task, shared, channel)
    
    try:
        # The key test is that this call must terminate.
        # We can add a timeout, but pytest-timeout is an external lib.
        # A well-behaved checker should finish this small state space quickly.
        s.run()
    except Exception as e:
        # An infinite loop would likely cause a timeout or memory error in a real runner.
        # Here, we just check that it doesn't raise an unrelated error.
        pytest.fail(f"Scheduler with cyclic program failed: {e}")

def test_api_and_results_unchanged_by_hashing():
    """
    Verifies that adding state hashing did not change the results for a known
    program with races and final states.
    """
    from solution import Lock

    s = Scheduler()
    shared = SharedObject()
    shared.value = 0

    # This program has a known data race.
    def writer(s_obj): s_obj.value = 1; yield
    def reader(s_obj): _ = s_obj.value; yield
        
    s.add(writer, shared)
    s.add(reader, shared)
    
    final_states, deadlocks, races = s.run()
    
    assert len(races) >= 1
    assert len(deadlocks) == 0
    
    # Check final states
    state_values = {s.value for s in final_states}
    # Path 1: writer -> reader. Final value 1.
    # Path 2: reader -> writer. Final value 1.
    # What if reader reads, writer writes? s.value=1.
    # What if writer writes, reader reads? s.value=1.
    # Let's make it more interesting:
    # Task A: v = s.v; s.v = v + 1
    # Task B: v = s.v; s.v = v + 1
    s2 = Scheduler()
    shared2 = SharedObject()
    shared2.value = 0
    lock = Lock()
    
    def unprotected_inc(s_obj):
        val = s_obj.value
        s_obj.value = val + 1
        yield
        
    s2.add(unprotected_inc, shared2)
    s2.add(unprotected_inc, shared2)
    
    final2, deadlocks2, races2 = s2.run()

    # The race is between the read and write in each task.
    assert len(races2) >= 1
    # Final values can be 1 (if one overwrites the other) or 2 (if serialized).
    # Hashing should not change this outcome.
    final_values2 = {s.value for s in final2}
    assert final_values2 == {1, 2}

In [ ]:
# test_solution_prompt_10.py
import pytest
from solution import Scheduler, Lock

def test_run_return_signature_is_unchanged():
    """Tests the return signature is still the 3-tuple."""
    s = Scheduler()
    def task(): yield
    s.add(task)
    result = s.run()
    assert isinstance(result, tuple) and len(result) == 3

def test_deadlock_trace_is_minimal():
    """
    Tests that a generated deadlock trace is minimal by creating a scenario
    with irrelevant preamble steps.
    """
    s = Scheduler()
    lock_a = Lock()
    lock_b = Lock()

    def task_1(lka, lkb):
        # Irrelevant preamble
        yield
        yield
        # Actual deadlock logic
        yield from lka.acquire()
        yield from lkb.acquire() # Blocks
        yield from lka.release()
        yield from lkb.release()

    def task_2(lka, lkb):
        # No preamble
        yield from lkb.acquire()
        yield from lka.acquire() # Blocks
        yield from lkb.release()
        yield from lka.release()

    s.add(task_1, lock_a, lock_b)
    s.add(task_2, lock_a, lock_b)
    
    _f, deadlocks, _r = s.run()
    
    assert len(deadlocks) == 1
    trace = deadlocks[0]
    
    # We can't know the exact trace format, but we can reason about its length.
    # A minimal trace to cause this deadlock requires 4 steps:
    # 1. T1 runs and acquires lock A.
    # 2. T2 runs and acquires lock B.
    # 3. T1 runs and tries to acquire lock B (blocks).
    # 4. T2 runs and tries to acquire lock A (blocks -> deadlock).
    # The trace length should be small (e.g., 4 scheduling decisions).
    # A non-minimal trace would include the two preamble yields from task_1.
    # This is hard to assert precisely without knowing the trace format.
    # Let's try to replay it and verify a property.
    
    # A good proxy for minimality is that the number of yields in the trace
    # is exactly what's needed for the deadlock.
    # We can try replaying a shorter version of the trace and see it fail.
    # This is too complex for a simple test.
    # Let's settle for a reasonable upper bound on the number of steps.
    
    # A simple trace could be a list of task IDs to run.
    # e.g., [0, 1, 0, 1]. The preamble would add more 0s at the start.
    # We expect the trace to NOT start with [0, 0, ...].
    if isinstance(trace, list):
         assert len(trace) <= 4, "Trace seems too long to be minimal"

def test_race_report_trace_is_minimal():
    """Tests that a data race report trace is also minimal."""
    from solution import SharedObject

    s = Scheduler()
    shared = SharedObject()
    shared.value = 0
    
    def writer(s_obj):
        # Irrelevant preamble
        yield
        yield
        # The racing write
        s_obj.value = 1
        yield

    def reader(s_obj):
        # The racing read
        _ = s_obj.value
        yield
        
    s.add(writer, shared)
    s.add(reader, shared)
    
    _f, _d, races = s.run()
    assert len(races) >= 1
    
    trace = races[0].trace if hasattr(races[0], 'trace') else races[0]

    # The minimal trace for this race only needs two steps:
    # 1. Run writer to the point of the write.
    # 2. Run reader to the point of the read.
    # Or vice-versa.
    # The preamble yields should be removed.
    if isinstance(trace, list):
        assert len(trace) <= 2, "Race trace seems too long to be minimal"

def test_no_bugs_produces_empty_lists():
    """Final check that a correct program yields no minimized bug reports."""
    from solution import Lock, SharedObject

    s = Scheduler()
    shared = SharedObject()
    shared.value = 0
    lock = Lock()
    
    def safe_task(s_obj, lk):
        yield from lk.acquire()
        s_obj.value += 1
        yield from lk.release()
        
    s.add(safe_task, shared, lock)
    s.add(safe_task, shared, lock)
    
    final_states, deadlocks, races = s.run()
    
    assert len(deadlocks) == 0
    assert len(races) == 0
    assert len(final_states) == 1
    assert final_states[0].value == 2